In [0]:
import pyspark.sql.functions as F

SCHEMA = "workspace.erp_lakehouse"

In [0]:
spark.table(f"{SCHEMA}.bronze_bc_items")\
    .select("*").display()

In [0]:
spark.table(f"{SCHEMA}.bronze_bc_items")\
    .filter("upper(displayName) like '%PARIS%'")\
    .select("number","displayName").display()

![image_1794368637534.png](../docs/images/image_1794368637534.png)

In [0]:
import pyspark.sql.functions as F

SCHEMA = "workspace.erp_lakehouse"

orders_b = spark.table(f"{SCHEMA}.bronze_b_orders")
lines_b = spark.table(f"{SCHEMA}.bronze_b_order_lines")

# 1. Earliest and latest file date across both tables
bounds = (orders_b.select("file_date")
    .union(lines_b.select("file_date"))
    .select(F.to_date("file_date").alias("d"))
    .agg(F.min("d").alias("start_day"), F.max("d").alias("end_day")))

# 2. One row per day between them
expected = (bounds
    .select(F.explode(F.sequence("start_day", "end_day")).alias("d"))
    .select(F.date_format("d", "yyyy-MM-dd").alias("file_date")))

# 3. Row counts per day from each table
orders_by_day = (orders_b.groupBy("file_date")
    .agg(F.count("*").alias("orders_rows")))
lines_by_day = (lines_b.groupBy("file_date")
    .agg(F.count("*").alias("lines_rows")))

max_orders = orders_by_day.agg(F.max("orders_rows")).first()[0]

# 4. Join the counts onto every expected day and set the status
load_check_b = (expected
    .join(orders_by_day, "file_date", "left")
    .join(lines_by_day, "file_date", "left")
    .fillna(0, ["orders_rows", "lines_rows"])
    .withColumn("status",
        F.when(F.col("orders_rows") == 0, "MISSING ORDERS FILE")
         .when(F.col("lines_rows") == 0, "MISSING LINES FILE")
         .when(F.col("orders_rows") < 0.5 * max_orders, "LOW VOLUME")
         .otherwise("OK"))
    .withColumn("checked_at", F.current_timestamp())
    .orderBy("file_date"))

load_check_b.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA}.load_check_b")
load_check_b.display()

![image_1794441037921.png](../docs/images/image_1794441037921.png)